# Clase 4 - Feature Engineering

Construimos nuevas variables sobre `competencia_01` siguiendo lo visto en `z402_Feature_Engineering_en_SQL.ipynb`: macros seguras (null / division por cero), combinacion de variables Visa/Master, ratios de riesgo, variables de actividad digital, conteo de productos, ranking dentro del mes (contra el data drifting por inflacion) y variables de tendencia temporal por cliente (lag, delta, media movil, pendiente), tanto sobre el valor nominal como sobre el rank (para depurar el efecto de la inflacion en la tendencia).


In [ ]:
%pip install jupysql
%pip install duckdb-engine


In [ ]:
import duckdb
import pandas as pd

%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

%sql duckdb://


In [ ]:
base_path = 'C:\\Users\\alejandro\\dmeyf2026\\monday\\'
#base_path = '/content/drive/MyDrive/DMEyF/2026/notebooks/'
dataset_path = base_path + 'data/'
dataset_file = 'competencia_01.csv'


In [ ]:
%%sql
create or replace table competencia_01 as
select
    *
from read_csv_auto("{{dataset_path + dataset_file}}")


## Macros seguras

`suma_sin_null` ya la vimos en la clase. `ratio_seguro` resuelve la tarea propuesta: hace un cociente que da `NULL` (en vez de error o `inf`) cuando el denominador es `NULL` o `0`, usando `nullif`.


In [ ]:
%%sql
CREATE OR REPLACE MACRO suma_sin_null(a, b) AS ifnull(a, 0) + ifnull(b, 0);
CREATE OR REPLACE MACRO ratio_seguro(a, b) AS a / nullif(b, 0);


## 1) Tarjetas combinadas (Visa + Master)

Unificamos el comportamiento de ambas tarjetas de credito en variables `tc_*`, para no duplicar la misma senal en dos columnas separadas.


In [ ]:
%%sql
select
    numero_de_cliente
    , foto_mes
    , suma_sin_null(Visa_msaldototal, Master_msaldototal) as tc_msaldototal
    , suma_sin_null(Visa_mconsumototal, Master_mconsumototal) as tc_mconsumototal
    , suma_sin_null(Visa_mlimitecompra, Master_mlimitecompra) as tc_mlimitecompra
    , ifnull(Visa_delinquency, 0) + ifnull(Master_delinquency, 0) as tc_cantidad_mora
    , least(Visa_Fvencimiento, Master_Fvencimiento) as tc_fvencimiento_menor
from competencia_01
limit 10


## 2) Ratios de riesgo y uso

Utilizacion del limite de credito, endeudamiento relativo al saldo, rentabilidad del mes vs. la anual, antiguedad relativa a la edad del cliente, comisiones vs. rentabilidad, y saldo vs. sueldo acreditado.


In [ ]:
%%sql
select
    numero_de_cliente
    , foto_mes
    , ratio_seguro(
        suma_sin_null(Visa_mconsumototal, Master_mconsumototal),
        suma_sin_null(Visa_mlimitecompra, Master_mlimitecompra)
      ) as r_uso_limite_tc
    , ratio_seguro(
        ifnull(mprestamos_personales, 0) + ifnull(mprestamos_prendarios, 0) + ifnull(mprestamos_hipotecarios, 0),
        mcuentas_saldo
      ) as r_endeudamiento_saldo
    , ratio_seguro(mrentabilidad, mrentabilidad_annual / 12.0) as r_rentabilidad_vs_anual
    , ratio_seguro(cliente_antiguedad, cliente_edad * 12) as r_antiguedad_vida
    , ratio_seguro(mcomisiones, mrentabilidad) as r_comisiones_vs_rentabilidad
    , ratio_seguro(mcuentas_saldo, mpayroll) as r_saldo_vs_sueldo
from competencia_01
limit 10


## 3) Actividad digital y 4) Cantidad de productos

Cuantos canales digitales usa el cliente, cuanta actividad genera por esos canales, y cuantos productos de inversion / seguro / prestamo tiene contratados.


In [ ]:
%%sql
select
    numero_de_cliente
    , foto_mes
    , ifnull(internet, 0) + ifnull(thomebanking, 0) + ifnull(tmobile_app, 0) + ifnull(tcallcenter, 0) as dig_canales_activos
    , ifnull(chomebanking_transacciones, 0) + ifnull(cmobile_app_trx, 0) + ifnull(ccallcenter_transacciones, 0) as dig_trx_total
    , ratio_seguro(
        ifnull(chomebanking_transacciones, 0) + ifnull(cmobile_app_trx, 0) + ifnull(ccallcenter_transacciones, 0),
        ctrx_quarter
      ) as dig_ratio_trx
    , ifnull(cseguro_vida, 0) + ifnull(cseguro_auto, 0) + ifnull(cseguro_vivienda, 0) + ifnull(cseguro_accidentes_personales, 0) as prod_total_seguros
    , ifnull(cinversion1, 0) + ifnull(cinversion2, 0) + ifnull(cplazo_fijo, 0) as prod_total_inversiones
    , ifnull(cprestamos_personales, 0) + ifnull(cprestamos_prendarios, 0) + ifnull(cprestamos_hipotecarios, 0) as prod_total_prestamos
from competencia_01
limit 10


## 5) Ranking dentro del mes (contra el data drifting por inflacion)

Los montos en pesos no son comparables entre meses distintos por la inflacion. En vez de (o ademas de) el valor absoluto, calculamos el `percent_rank` de cada cliente dentro de su propio `foto_mes` para las variables monetarias clave: esa posicion relativa si es comparable mes a mes.


In [ ]:
campos_rank = [
    'mrentabilidad',
    'mcuentas_saldo',
    'mtarjeta_visa_consumo',
    'mtarjeta_master_consumo',
    'mpayroll',
    'ctrx_quarter',
]

features_rank = ""
for campo in campos_rank:
    features_rank += f"\n    , percent_rank() over (partition by foto_mes order by {campo}) as rank_{campo}"

print(features_rank)


## 6) Tendencia temporal por cliente

Para detectar el "enfriamiento" de un cliente antes de la baja, calculamos por cliente (ventana ordenada por `foto_mes`): el valor del mes anterior (`lag`), el delta contra el mes anterior, la media movil de 3 meses y la pendiente (`regr_slope`) de los ultimos 3 meses, para variables de actividad y de plata.


In [ ]:
campos_tendencia = [
    'ctrx_quarter',
    'mcuentas_saldo',
    'mtarjeta_visa_consumo',
    'mtarjeta_master_consumo',
    'mpayroll',
    'mrentabilidad',
    'cpayroll_trx',
]

features_tendencia = ""
for campo in campos_tendencia:
    features_tendencia += f"""
    , lag({campo}, 1) over ventana_cliente as lag_1_{campo}
    , {campo} - lag({campo}, 1) over ventana_cliente as delta_1_{campo}
    , avg({campo}) over ventana_3m as ma_3_{campo}
    , regr_slope({campo}, cliente_antiguedad) over ventana_3m as slope_3_{campo}"""

print(features_tendencia)


## 6b) Tendencia libre de inflacion (sobre el ranking)

Las tendencias del punto 6 se calculan sobre el valor nominal en pesos, asi que una parte del `delta`/`slope` puede ser pura inflacion acumulada entre meses, no un cambio real de comportamiento. Repetimos las mismas 4 metricas (`lag`, `delta`, media movil, `slope`) pero sobre el `rank_*` del punto 5 en vez del valor nominal: como el rank ya esta corregido por la distribucion de cada mes, su tendencia refleja mejor si el cliente mejoro/empeoro *en relacion a los demas*, sin el ruido de la inflacion.


In [ ]:
campos_tendencia_rank = campos_rank

features_tendencia_rank = ""
for campo in campos_tendencia_rank:
    features_tendencia_rank += f"""
    , lag(rank_{campo}, 1) over ventana_cliente as lag_1_rank_{campo}
    , rank_{campo} - lag(rank_{campo}, 1) over ventana_cliente as delta_1_rank_{campo}
    , avg(rank_{campo}) over ventana_3m as ma_3_rank_{campo}
    , regr_slope(rank_{campo}, cliente_antiguedad) over ventana_3m as slope_3_rank_{campo}"""

print(features_tendencia_rank)


## Query final: armamos la tabla con todas las features nuevas

Une los bloques 1 a 4 (escritos a mano, son pocas variables) con los bloques 5, 6 y 6b (generados por loop, interpolados con `{{ }}` como en la clase). El ranking del punto 5 se calcula primero en un CTE (`base_rank`), porque para poder calcularle `lag`/`delta`/`slope` en el punto 6b necesita existir como columna real, no solo como alias dentro del mismo `select`.


In [ ]:
%%sql
create or replace table competencia_01_fe as
with base_rank as (
    select
        c.*
        {{features_rank}}
    from competencia_01 c
)
select
    br.*

    -- 1) tarjetas combinadas (Visa + Master)
    , suma_sin_null(Visa_msaldototal, Master_msaldototal) as tc_msaldototal
    , suma_sin_null(Visa_mconsumototal, Master_mconsumototal) as tc_mconsumototal
    , suma_sin_null(Visa_mlimitecompra, Master_mlimitecompra) as tc_mlimitecompra
    , suma_sin_null(Visa_mpagado, Master_mpagado) as tc_mpagado
    , suma_sin_null(Visa_cconsumos, Master_cconsumos) as tc_cconsumos
    , suma_sin_null(Visa_cadelantosefectivo, Master_cadelantosefectivo) as tc_cadelantosefectivo
    , ifnull(Visa_delinquency, 0) + ifnull(Master_delinquency, 0) as tc_cantidad_mora
    , least(Visa_Fvencimiento, Master_Fvencimiento) as tc_fvencimiento_menor

    -- 2) ratios de riesgo / uso
    , ratio_seguro(
        suma_sin_null(Visa_mconsumototal, Master_mconsumototal),
        suma_sin_null(Visa_mlimitecompra, Master_mlimitecompra)
      ) as r_uso_limite_tc
    , ratio_seguro(
        ifnull(mprestamos_personales, 0) + ifnull(mprestamos_prendarios, 0) + ifnull(mprestamos_hipotecarios, 0),
        mcuentas_saldo
      ) as r_endeudamiento_saldo
    , ratio_seguro(mrentabilidad, mrentabilidad_annual / 12.0) as r_rentabilidad_vs_anual
    , ratio_seguro(cliente_antiguedad, cliente_edad * 12) as r_antiguedad_vida
    , ratio_seguro(mcomisiones, mrentabilidad) as r_comisiones_vs_rentabilidad
    , ratio_seguro(mcuentas_saldo, mpayroll) as r_saldo_vs_sueldo

    -- 3) actividad digital
    , ifnull(internet, 0) + ifnull(thomebanking, 0) + ifnull(tmobile_app, 0) + ifnull(tcallcenter, 0) as dig_canales_activos
    , ifnull(chomebanking_transacciones, 0) + ifnull(cmobile_app_trx, 0) + ifnull(ccallcenter_transacciones, 0) as dig_trx_total
    , ratio_seguro(
        ifnull(chomebanking_transacciones, 0) + ifnull(cmobile_app_trx, 0) + ifnull(ccallcenter_transacciones, 0),
        ctrx_quarter
      ) as dig_ratio_trx

    -- 4) cantidad de productos
    , ifnull(cseguro_vida, 0) + ifnull(cseguro_auto, 0) + ifnull(cseguro_vivienda, 0) + ifnull(cseguro_accidentes_personales, 0) as prod_total_seguros
    , ifnull(cinversion1, 0) + ifnull(cinversion2, 0) + ifnull(cplazo_fijo, 0) as prod_total_inversiones
    , ifnull(cprestamos_personales, 0) + ifnull(cprestamos_prendarios, 0) + ifnull(cprestamos_hipotecarios, 0) as prod_total_prestamos

    -- 6) tendencia temporal por cliente (valor nominal)
    {{features_tendencia}}

    -- 6b) tendencia temporal por cliente (sobre el rank, sin efecto inflacion)
    {{features_tendencia_rank}}

from base_rank br
window
    ventana_cliente as (partition by numero_de_cliente order by foto_mes),
    ventana_3m as (partition by numero_de_cliente order by foto_mes rows between 3 preceding and current row)


In [ ]:
%%sql
select * from competencia_01_fe limit 5


## Guardar el nuevo df


In [ ]:
%%sql
COPY competencia_01_fe TO "{{dataset_path + 'competencia_01_fe.csv'}}" (FORMAT CSV, HEADER TRUE);
